# Deep Learning for Text Data, HSE FCS
## Homework assignment 3: Model Size Reduction
### Assessment and penalties

The maximum grade allowed for this assignment is __16 points__.

The assignment must be completed independently. "Similar" solutions are considered plagiarism, and all students involved (including those from whom the plagiarism was committed) cannot receive more than 0 points for it. All code must be written independently. Using someone else's code is prohibited, even with a link to the source. Within reasonable limits, of course. Taking a couple of obvious lines of code to implement a small feature is acceptable.

Ineffective code implementation may negatively impact your grade. Your grade may also be reduced for poorly readable code and poorly formatted plots. All answers must be accompanied by code or comments explaining how they were obtained.

__Soft deadline: 22.11.26 23:59__ \
__Hard deadline: 24.11.26 23:59__

### About task

In this assignment, you'll learn to solve the Named Entity Recognition (NER) task on the most popular dataset – [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). You'll be given a pretrained BERT, which you'll need to shrink to 20M parameters with minimal quality loss. To do this, you'll implement embedding factorization, distillation, parameter sharing, and so on by yourself.

In this assignment, you'll have to run quite a lot of experiments, so we recommend not writing all the code in the notebook, but instead creating separate files for individual logical blocks and organizing everything as a project. This will keep your notebook from growing too large and will make things much easier for both you and the graders. Also, try to log all your experiments to wandb or other platforms so that nothing gets lost.

In this assignment, __using agents to write code is prohibited__, but you may use them to help with reading papers and understanding methods.

### Assessment
The grade for this homework will consist of the grade for the __tasks__ and for the __report__, in which you are required to describe the work you've done. You can get up to 2 points for the report; however, if there is no report, points for the corresponding tasks will not be awarded. The tasks are divided into two parts: _numbered_ and _elective_. For the _numbered_ tasks you can get 6 points in total, and for the _elective_ tasks you can get up to 8. That is, you can get 16 points for the entire homework (but don't celebrate too early, it's not that easy). Everything you score above 10 will count as bonus points.

### About dataset

Named Entity Recognition is the task of classifying tokens into entity classes. CoNLL-2003 uses **BIO** tagging (Beginning, Inside, Outside) to label entities, where the labels mean the following:

- *B-{label}* – beginning of the *{label}* entity
- *I-{label}* – continuation of the *{label}* entity
- *O* – not an entity

There are also other tagging schemes, for example, BILUO. You can read about them [here](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) and [here](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

There are 9 different labels in the dataset in total.
- O – the word doesn't correspond to any entity.
- B-PER/I-PER – the word or set of words corresponds to a specific _person_.
- B-ORG/I-ORG – the word or set of words corresponds to a specific _organization_.
- B-LOC/I-LOC – the word or set of words corresponds to a specific _location_.
- B-MISC/I-MISC – the word or set of words corresponds to an entity that doesn't belong to any of the previous ones. For example, a nationality, a work of art, an event, etc.

Let's get started!

Let's start by loading and preprocessing the dataset.

In [1]:
from datasets import load_dataset

dataset = load_dataset("ashaba1in/conll2003")

dataset = dataset.remove_columns(["id", "pos_tags", "chunk_tags"])
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

In [13]:
from datasets import load_dataset

dataset2 = load_dataset("hgissbkh/conll2003-en")
dataset2

DatasetDict({
    train: Dataset({
        features: ['words', 'ner'],
        num_rows: 14042
    })
    validation: Dataset({
        features: ['words', 'ner'],
        num_rows: 3252
    })
    test: Dataset({
        features: ['words', 'ner'],
        num_rows: 3454
    })
})

In [16]:
dataset['train'][1]

{'tokens': ['Peter', 'Blackburn'], 'ner_tags': [1, 2]}

In [2]:
dataset['train'][0]

{'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [3]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [4]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


### Preprocessing

Throughout this homework assignment, we'll use the _cased_ version of BERT, meaning that the tokenizer will take the case of words into account. Case matters for the NER task, since names of people, organizations, or works of art are often capitalized, and it would be silly to hide such information from the model.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

During tokenization, words may be split into several tokens (like the word `Fischler` in the example below), which creates a mismatch between the number of tokens and labels. We'll have to fix this mismatch manually.

In [6]:
example = dataset["train"][12]
words = example["tokens"]
tags = [label_names[t] for t in example["ner_tags"]]
tokenized_text = tokenizer(example["tokens"], is_split_into_words=True)

print('Words: ', words)
print('Tokens:', tokenized_text.tokens())
print('Labels:', tags)

Words:  ['Only', 'France', 'and', 'Britain', 'backed', 'Fischler', "'s", 'proposal', '.']
Tokens: ['[CLS]', 'Only', 'France', 'and', 'Britain', 'backed', 'Fi', '##sch', '##ler', "'", 's', 'proposal', '.', '[SEP]']
Labels: ['O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'O', 'O']


__Task 1 (1 point).__ Tokenize the entire dataset and, for each text, align the tokens with the labels so that each token corresponds to exactly one label. It's important to preserve the BIO notation. And don't forget about the special tokens! You should get something like this:

In [9]:
aligned_labels = align_labels_with_tokens(example["ner_tags"], tokenized_text)
tags = [label_names[t] if t > -1 else t for t in aligned_labels]
print("Aligned labels:", aligned_labels)
print("Aligned label names:", tags)

Aligned labels: [-100    0    5    0    5    0    1    2    2    0    0    0    0 -100]
Aligned label names: [-100, 'O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', -100]


In [ ]:
# your code here

### Metric

To evaluate NER quality, the micro-averaged F1 score is typically used. We'll load it from the `seqeval` library. The `f1_score` function takes two 2D lists of true and predicted labels written as text and returns the F1 value for them. You can use it with the default parameters.

In [12]:
# ! pip install seqeval

In [46]:
from seqeval.metrics import f1_score

A peculiarity of computing F1 for NER is that in some situations incorrect answers can be counted as correct. For example, if the model predicted `['I-PER', 'I-PER']`, we can guess that it should actually be `['B-PER', 'I-PER']`, since an entity can't start with `I-`. The `f1_score` function takes this into account and therefore only works with text representations of labels.

### Model

We'll use `bert-base-cased` as the base model. As you understand, it wasn't trained for the NER task. Therefore, before we start reducing the size of BERT, it needs to be fine-tuned.

__Task 2 (1 point).__ Fine-tune `bert-base-cased` on our dataset using regular fine-tuning. You should get at least 0.9 F1 on the test set. Note that the higher the quality of the large model, the better the distilled student will perform. You can use `Trainer` from Hugging Face for training.

In [43]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(label_names))

print('Number of parameters:', sum(p.numel() for p in model.parameters()))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of parameters: 107726601


In [44]:
# your code here

### Embedding Matrix Factorization

You may notice that the embedding matrix currently takes up $V \cdot H = 28996 \cdot 768 = 22{,}268{,}928$ parameters. That's a whole fifth of the entire model! Let's try to do something about it. The [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) model proposes factorizing the embedding matrix into the product of two small matrices. This way, the embedding parameters will contain $V \cdot E + E \cdot H$ elements, which is much less than $V \cdot H$ if $H \gg E$. The authors choose $E = 128$, but nothing prevents us from taking any other value. For example, by choosing $H = 64$, we'll reduce the number of parameters by about 20M.

__Task 3 (1 point).__ Write a wrapper class for the embedding layer that implements factorization into two matrices, and fine-tune the factorized model. Note that both matrices can be initialized using SVD (or better yet, [FWSVD](https://arxiv.org/pdf/2207.00112)!), so that the initial approximation is good. This will save a lot of fine-tuning time. With a decomposition rank of $H = 64$, you should get an F1 above 0.87.

In [ ]:
# your code here

### Knowledge Distillation

Knowledge distillation is a training paradigm in which the knowledge of a teacher model is distilled into a student model. The student can be an arbitrary smaller model solving the same task, but usually the student has the same architecture as the teacher. Distillation uses two loss functions:

1. Standard cross-entropy.
1. A function defining the distance between the teacher's and student's prediction distributions. KL divergence is used most often.

To keep the teacher's prediction distribution from being degenerate, a temperature greater than 1, for example, 2 or 5, is added to the softmax.
__Important:__ when dividing the logits by the temperature, the gradient values decrease by a factor of $\tau^2$ (check this!). Therefore, to bring them back to the original scale, the loss must be multiplied by $\tau^2$. You can read more about this in Section 2.1 of the [original paper](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Task 4 (3 points).__ Implement the knowledge distillation method shown in the picture. To compute the loss between the student's and teacher's predictions, use KL divergence [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (pay attention to the format of its inputs). To get the final loss, sum the soft loss and the hard loss.
Use the fine-tuned BERT from Task 2 as the teacher. As the student, take an untrained model with __no more than 20M__ parameters. You can use embedding matrix factorization to reduce the number of parameters. If you did everything correctly, you should get an F1 of at least 0.7 on the test set. About 20k training iterations should be enough for this. If something isn't working, you can refer to the [DistilBERT](https://arxiv.org/abs/1910.01108) paper and [this paper](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).

__Important:__
* Don't forget to add _warmup_ when training the student.
* Don't forget to switch the teacher to _eval_ mode.

In [ ]:
# your code here

# Elective tasks

As you understand, there are quite a lot of other ways to reduce the size of a trained model. In this section, you're invited to implement various techniques of your choice. Each of them is worth a different number of points depending on its difficulty. Your implementation will be assessed both by the code and by the quality on the test set. All points for this homework that you score above 10 will count as bonus points.
In Task 4, you trained a model with a parameter limit of 20M. When implementing techniques from this section, stick to the same limit. This will allow you to compare the methods with each other fairly and draw correct conclusions. Describe everything you tried in the report.

* __Weight sharing (1 point).__ [ALBERT](https://arxiv.org/pdf/1909.11942), a modification of BERT, proposes sharing weights between layers in addition to embedding factorization. That is, different layers use the same weights. This technique is equivalent to applying the same layer several times. It allows reducing the number of parameters several times over without losing much quality.
* __Weight sharing++ (0.5 points).__ Instead of simply applying the same layers, you can fine-tune a LoRA for each layer so that the layers can specialize. This is exactly what is done in [this paper](https://arxiv.org/pdf/2410.20672) from DeepMind.
* __Intermediate layer matching (1.5 points).__ We discussed that, in addition to bringing the outputs of the student model closer to the outputs of the teacher model, you can also do the same for the outputs of intermediate layers. [This paper](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) describes in detail how this can be done. Another __+1 point__ for implementing the method from [MiniLMv2](https://arxiv.org/pdf/2012.15828).
* __Pruning (3 points).__ The [SparseGPT](https://arxiv.org/abs/2301.00774) method proposes an approach that removes model weights once after training. It turns out that it's possible to remove up to half of all weights without any loss in quality. The math behind the technique is rather complex, but the general approach is simple – we remove weights in each layer separately, and when part of a layer's weights is removed, the remaining weights are readjusted so that the overall output of the layer doesn't change. As a baseline, implement the simpler [Wanda](https://arxiv.org/pdf/2306.11695) method.
* __Student initialization via pruning (2 points).__ Training a student from scratch takes quite a lot of time. Instead, the student can be initialized by pruning the teacher and then fine-tuned using distillation. This was first done in [Minitron](https://arxiv.org/pdf/2408.11796), and later in Ministral 3, Llama 3.2, Falcon3, etc. An important problem with this approach is that layers may be pruned unevenly, which makes it harder to distribute the load when using the model. But if the task is simply to reduce the size, this is a great approach.

